In [18]:
import pandas as pd
from scipy.stats import zscore
import seaborn as sns
from scipy.cluster.hierarchy import fcluster
import numpy as np

In [19]:
header = ['Naive 1', 'Naive 2', 'Naive 3', 'Naive 4', 'Early Pre-TFH 1', 'Early Pre-TFH 2', 'Early Pre-TFH 3', 'Early Pre-TFH 4', 'Late Pre-TFH 1', 'Late Pre-TFH 2', 'Late Pre-TFH 3', 'Late Pre-TFH 4', 'GC 1', 'GC 2', 'GC 3', 'GC 4']

rna_seq = pd.read_csv('/ix/djishnu/Alisa/Tfh/testing_for_github/Data/Input_data/rna_normalized_data_matrix.tsv', skiprows = [0], names=header, sep ='\t')
rna_seq = rna_seq.reset_index()

rna_seq = rna_seq.rename(columns={'index': "Gene"})
rna_seq[['Gene_ID', 'Genes']] = rna_seq.Gene.str.split("_",n=1, expand=True)

In [20]:
rna_seq['ATAC_Naive'] = rna_seq[['Naive 1', 'Naive 2', 'Naive 3', 'Naive 4']].mean(axis=1) + 0.000001
rna_seq['ATAC_Early_Pre-TFH'] = rna_seq[['Early Pre-TFH 1', 'Early Pre-TFH 2', 'Early Pre-TFH 3', 'Early Pre-TFH 4']].mean(axis=1) + 0.000001
rna_seq['ATAC_Late_Pre-TFH'] = rna_seq[['Late Pre-TFH 1', 'Late Pre-TFH 2', 'Late Pre-TFH 3', 'Late Pre-TFH 4']].mean(axis=1) + 0.000001
rna_seq['ATAC_GC'] = rna_seq[['GC 1', 'GC 2', 'GC 3', 'GC 4']].mean(axis=1) + 0.000001
rna_seq_mean = rna_seq[['Genes', 'ATAC_Naive','ATAC_Early_Pre-TFH', 'ATAC_Late_Pre-TFH', 'ATAC_GC']]

In [21]:
rna_seq_mean = rna_seq_mean.rename(columns={'ATAC_Naive':'Naive','ATAC_Early_Pre-TFH':'Early_Extra_GC_Tfh', 'ATAC_Late_Pre-TFH':'Late_Extra_GC_Tfh', 'ATAC_GC':'GC Tfh'})

In [22]:
rna_seq_mean = rna_seq_mean.set_index('Genes')

In [23]:
rna_seq_mean.to_csv('/ix/djishnu/Alisa/Tfh/testing_for_github/Data/Input_data/rna_seq_mean.csv')

In [6]:
rna_seq_mean['LogChange'] = np.log2(np.divide(rna_seq_mean["ATAC_GC"],rna_seq_mean["ATAC_Early_Pre-TFH"]))
rna_seq_mean = rna_seq_mean.reset_index()

#rna_seq_mean_filtered = rna_seq_mean[rna_seq_mean['total'] > threshold]
#print(rna_seq_mean_filtered)
rna_seq_sign_change = rna_seq_mean[['Genes', 'LogChange']]
print(rna_seq_sign_change)

               Genes  LogChange
0             TSPAN6 -17.554026
1               TNMD   0.000000
2               DPM1  -0.131166
3              SCYL3   2.485725
4           C1orf112  -2.487404
...              ...        ...
60670  CTD-2060L22.1 -16.784421
60671   RP11-107E5.4   0.000000
60672          HYMAI  20.806715
60673     RARRES2P11   0.000000
60674   RP11-299P2.2   0.000000

[60675 rows x 2 columns]


In [7]:
rna_seq_sign_change.to_csv('/ix/djishnu/Alisa/Tfh/testing_for_github/Data/Input_data/rna_earlyvsGC_log2FC.csv')

In [8]:
#Cap the LogFC values at abs(2):
rna_seq_sign_change = rna_seq_sign_change.copy()
rna_seq_sign_change['LogChange_Thresh'] = rna_seq_sign_change['LogChange'].clip(lower=-2, upper=2)


In [9]:
rna_seq_sign_change.to_csv('/ix/djishnu/Alisa/Tfh/testing_for_github/Data/Input_data/rna_earlyvsGC_Thresh_log2FC.csv')

In [10]:
rna_seq_filtered = rna_seq_mean.copy()

# Compute row means only on numeric columns
rna_seq_filtered["total"] = rna_seq_filtered.mean(axis=1, numeric_only=True)

# Calculate the 10% threshold
threshold = rna_seq_filtered["total"].quantile(0.10)

# Filter out the lowest 10%
filtered_df = rna_seq_filtered[rna_seq_filtered["total"] > threshold]

In [11]:
rna_seq_mean['LogChange'] = np.log2(np.divide(rna_seq_mean["ATAC_GC"],rna_seq_mean["ATAC_Late_Pre-TFH"]))
rna_seq_mean = rna_seq_mean.reset_index()

rna_seq_sign_change = rna_seq_mean[['Genes', 'LogChange']]
print(rna_seq_sign_change)

               Genes  LogChange
0             TSPAN6 -19.323378
1               TNMD   0.000000
2               DPM1   0.715762
3              SCYL3   2.204353
4           C1orf112  -0.080361
...              ...        ...
60670  CTD-2060L22.1 -19.226702
60671   RP11-107E5.4   0.000000
60672          HYMAI  20.806715
60673     RARRES2P11   0.000000
60674   RP11-299P2.2   0.000000

[60675 rows x 2 columns]


In [12]:
rna_seq_sign_change.to_csv('/ix/djishnu/Alisa/Tfh/testing_for_github/Data/Input_data/rna_latevsGC_log2FC.csv')

In [13]:
rna_seq_sign_change = rna_seq_sign_change.copy()
rna_seq_sign_change['LogChange_Thresh'] = rna_seq_sign_change['LogChange'].clip(lower=-2, upper=2)

In [14]:
rna_seq_sign_change.to_csv('/ix/djishnu/Alisa/Tfh/testing_for_github/Data/Input_data/rna_latevsGC_Thresh_log2FC.csv')